In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

RAW_PATH = "../data/Nassau_Candy_Distributor.csv"
OUT_PATH = "../outputs/cleaned_data.csv"

LEAD_TIME_RANGES = {
    "Same Day": (0, 1),
    "First Class": (1, 3),
    "Second Class": (3, 5),
    "Standard Class": (5, 8),
}

In [4]:
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(10194, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [2]:
def simulate_lead_time(ship_mode_series):
    lead_times = np.zeros(len(ship_mode_series))

    for mode, (lo, hi) in LEAD_TIME_RANGES.items():
        mask = (ship_mode_series == mode).values
        n = mask.sum()
        if n == 0:
            continue
        mode_point = lo + (hi - lo) * 0.3
        sampled = np.random.triangular(lo, mode_point, hi, n)
        lead_times[mask] = sampled

    noise = np.random.normal(0, 0.4, len(lead_times))
    lead_times = np.clip(lead_times + noise, 0, 10)
    return np.round(lead_times, 2)

In [5]:
df["Lead Time"] = simulate_lead_time(df["Ship Mode"])
df.groupby("Ship Mode")["Lead Time"].describe()

,count,mean,std,min,25%,50%,75%,max
Ship Mode,,,,,,,,
First Class,1548.0,1.866796,0.566082,0.00,1.47,1.83,2.2425,3.89
Same Day,547.0,0.461993,0.380977,0.00,0.12,0.43,0.7100,1.94
Second Class,1979.0,3.875336,0.583682,2.19,3.48,3.86,4.2800,5.87
Standard Class,6120.0,6.284248,0.736778,4.11,5.76,6.23,6.7900,8.74
